# Disaster XAI-Net — News & Social-Media Module (CrisisMMD)

**Purpose.** The satellite pipeline (xBD, ConvNeXt/ViT siamese) needs *pre + post overhead image pairs plus building footprints*. A TV/news/social-media picture is a single, ground-level, oblique photo with no "before" image, so it **cannot** be fed into the satellite model. This notebook builds the *second, independent information source* of the system: models that read disaster news/social-media **text** and **images**, extract **locations**, and produce a per-item report that can later be joined with the satellite damage map at the location level.

```
 news / social-media item (text and/or image)
            │
   ┌────────┼──────────────────────────────┬──────────────────────────┐
   │ TEXT (transformer)                    │ IMAGE (ConvNeXt-Tiny)     │
   │  • informative vs not                 │  • informative vs not     │
   │  • humanitarian category (5)          │  • humanitarian (5)       │
   │  • location extraction (NER)          │  • damage severity (3)    │
   │  • hazard keywords (rule-based)       │                           │
   └────────┬──────────────────────────────┴────────────┬─────────────┘
            └──────────── per-item report (+ heuristic priority) ────┘
                                   │
                    joined with the satellite map by location (demo)
```

**What is evaluated here (goes into the paper as measured results)**
* Text informativeness and humanitarian-category classification (transformer, 3 seeds) vs TF-IDF + logistic-regression and majority baselines.
* Image informativeness, humanitarian category and **damage severity** (ConvNeXt-Tiny).
* All on the **official CrisisMMD v2.0 train/dev/test split**; dev is used for model selection, test is scored once per model. Per-event breakdowns are reported.

**What is NOT quantitatively evaluated (must be described as such in the paper)**
* Location extraction (no gold labels in CrisisMMD). A 60-tweet manual-check sheet is exported so the team can measure precision in ~20 minutes (section 10b).
* The rule-based hazard keywords and the heuristic per-item priority score.
* The link "news item → satellite tiles" — shown as a **case study** on test-split items only (section 11), and the replay feed for the demo is produced by the *real* models.

Checkpoints + resume: every (task, seed) writes a completion marker; re-running skips finished runs. Image training also resumes mid-run from the last epoch.

---
## 1. Imports and configuration

The `CNS_*` environment variables exist only for an offline smoke test of the code; on Kaggle nothing needs to be set. **Kaggle settings: GPU on, Internet on.**

In [ ]:
import os, re, io, gc, json, time, math, random, hashlib, shutil, zipfile, tarfile, glob, warnings
import datetime, platform, sys, csv, importlib.util, urllib.request, urllib.parse
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile
import timm
from torchvision import transforms
import sklearn
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef,
                             cohen_kappa_score, confusion_matrix, classification_report, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
transformers.logging.set_verbosity_error()

_env = os.environ.get
SMOKE = _env("CNS_SMOKE", "0") == "1"          # offline code test with synthetic data - never on Kaggle

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()
print("device:", device, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| timm", timm.__version__)

# ---------------- models ----------------
TEXT_MODEL   = _env("CNS_TEXT_MODEL", "distilroberta-base")   # swap for "roberta-base" if time allows
IMG_MODEL    = _env("CNS_IMG_MODEL", "convnext_tiny")          # same family as the satellite model
NER_MODEL    = "dslim/bert-base-NER"                           # pretrained, not fine-tuned here

# ---------------- protocol ----------------
TEXT_SEEDS   = [int(s) for s in _env("CNS_TEXT_SEEDS", "42,43,44").split(",")]
IMG_SEEDS    = [int(s) for s in _env("CNS_IMG_SEEDS", "42").split(",")]   # add 43,44 if GPU time allows
TEXT_EPOCHS, TEXT_LR, TEXT_MAXLEN, TEXT_BATCH, TEXT_WARMUP, TEXT_PATIENCE = int(_env("CNS_TEXT_EPOCHS", 4)), 2e-5, 128, 32, 0.1, 2
IMG_SIZE     = int(_env("CNS_IMG_SIZE", 224))
IMG_EPOCHS, IMG_PATIENCE, IMG_LR, IMG_BATCH = int(_env("CNS_IMG_EPOCHS", 15)), 4, 1e-4, int(_env("CNS_IMG_BATCH", 32))
WEIGHT_DECAY, LABEL_SMOOTH = 1e-4, 0.05
NUM_WORKERS  = int(_env("CNS_NUM_WORKERS", 2))
DROP_TRAIN_LEAKS = True        # drop TRAIN rows that duplicate a dev/test tweet or image (dev/test never altered)

# which tasks to run (remove one to save GPU time)
RUN_TASKS = _env("CNS_TASKS", "text_informative,text_humanitarian,image_damage,image_humanitarian,image_informative").split(",")

RESUME, FORCE_RETRAIN = True, False
GEOCODE = (not SMOKE) and True   # OpenStreetMap Nominatim lookups for the case study (1 request/s, cached)
CASE_STUDY_EVENTS = ['hurricane_harvey', 'mexico_earthquake', 'california_wildfires']  # 2017 events also present in xBD - verify against your satellite tiles

# ---------------- classes ----------------
HUM5 = ['affected_injured_or_dead_people', 'infrastructure_and_utility_damage', 'not_humanitarian',
        'other_relevant_information', 'rescue_volunteering_or_donation_effort']
TASKS = {
    'text_informative':   dict(modality='text',  family='informative',  classes=['not_informative', 'informative'], ordinal=False),
    'text_humanitarian':  dict(modality='text',  family='humanitarian', classes=HUM5, ordinal=False),
    'image_damage':       dict(modality='image', family='damage',       classes=['little_or_no_damage', 'mild_damage', 'severe_damage'], ordinal=True),
    'image_humanitarian': dict(modality='image', family='humanitarian', classes=HUM5, ordinal=False),
    'image_informative':  dict(modality='image', family='informative',  classes=['not_informative', 'informative'], ordinal=False),
}
TASKS = {k: v for k, v in TASKS.items() if k in RUN_TASKS}
SPLITS = ['train', 'dev', 'test']

# ---------------- output ----------------
OUT = _env("CNS_OUTPUT_ROOT", "/kaggle/working/crisis_news_results")
DATA_ROOT = _env("CNS_DATA_ROOT") or ("/kaggle/temp/crisismmd" if os.path.isdir("/kaggle") else os.path.abspath("_crisismmd_data"))
CACHE_DIR = os.path.join(os.path.dirname(DATA_ROOT.rstrip('/')), "crisismmd_img_cache")
D = {k: os.path.join(OUT, v) for k, v in {
    'ckpt': 'checkpoints', 'metrics': 'metrics', 'preds': 'predictions', 'plots': 'plots',
    'data': 'data_report', 'bundle': 'news_bundle', 'case': 'case_study', 'repro': 'reproducibility'}.items()}
for p in list(D.values()) + [DATA_ROOT, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

def class_weights(y, n):
    """'balanced' weights that stay finite if a class is absent from y."""
    c = np.bincount(np.asarray(y, dtype=int), minlength=n).astype(float)
    return len(y) / (n * np.maximum(c, 1.0))

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, np.bool_): return bool(o)
    return str(o)

def jdump(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    os.replace(tmp, path)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def tload(path):
    try: return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError: return torch.load(path, map_location="cpu")

HYPERPARAMS = dict(text_model=TEXT_MODEL, img_model=IMG_MODEL, ner_model=NER_MODEL, text_seeds=TEXT_SEEDS,
                   img_seeds=IMG_SEEDS, text_epochs=TEXT_EPOCHS, text_lr=TEXT_LR, text_maxlen=TEXT_MAXLEN,
                   text_batch=TEXT_BATCH, text_warmup=TEXT_WARMUP, text_patience=TEXT_PATIENCE, img_size=IMG_SIZE,
                   img_epochs=IMG_EPOCHS, img_patience=IMG_PATIENCE, img_lr=IMG_LR, img_batch=IMG_BATCH,
                   weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTH, drop_train_leaks=DROP_TRAIN_LEAKS,
                   selection_metric='dev macro-F1', tasks=list(TASKS), amp=AMP, smoke=SMOKE)
jdump(HYPERPARAMS, os.path.join(D['repro'], "hyperparameters.json"))
print("tasks:", list(TASKS)); print("results ->", OUT); print("data    ->", DATA_ROOT)

---
## 2. Data — CrisisMMD v2.0 (official split)

Primary source: the Hugging Face copy published by the dataset authors (`QCRI/CrisisMMD`, subsets `informative`, `humanitarian`, `damage`, each with `train/dev/test`; one row per **image**, with the tweet text alongside). Fallback: the official CrisisNLP downloads. Text-level labels are taken from a `label_text` column when the source provides one; otherwise from the official split files; only as a last resort are they derived from image labels (flagged loudly, because that is weaker supervision).

License: CC BY-NC-SA 4.0 — cite Alam, Ofli & Imran (ICWSM 2018) and Ofli, Alam & Imran (ISCRAM 2020).

In [ ]:
HF_REPO = "QCRI/CrisisMMD"
OFFICIAL_SPLIT_URL = "https://crisisnlp.qcri.org/data/crisismmd/crisismmd_datasplit_all.zip"
OFFICIAL_DATA_URL  = "https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz"
FAMILIES = sorted({t['family'] for t in TASKS.values()} | {'informative'})   # informative table = all images
OFFICIAL_DIR = os.path.join(DATA_ROOT, "official_splits")


def make_smoke_data(root):
    """Synthetic stand-in with the same schema (offline code test only)."""
    rng = np.random.RandomState(0)
    events = ['hurricane_harvey', 'mexico_earthquake', 'california_wildfires']
    places = {'hurricane_harvey': ['Houston', 'Texas', 'Rockport'], 'mexico_earthquake': ['Mexico City', 'Puebla'],
              'california_wildfires': ['Santa Rosa', 'Napa', 'Sonoma']}
    words = "damage flood rescue help people dead injured roof house road destroyed donate volunteers power storm fire quake".split()
    hum8 = ['affected_individuals', 'infrastructure_and_utility_damage', 'injured_or_dead_people', 'missing_or_found_people',
            'not_humanitarian', 'other_relevant_information', 'rescue_volunteering_or_donation_effort', 'vehicle_damage']
    tid = 905000000000000000
    for split, n in [('train', 90), ('dev', 30), ('test', 30)]:
        rows = []
        for i in range(n):
            tid += int(rng.randint(10**9, 10**12)); ev = events[i % 3]
            text = f"RT @news{i%5}: " + " ".join(rng.choice(words, 8)) + f" in {rng.choice(places[ev])} #{ev.split('_')[0].title()} https://t.co/x{i}"
            n_img = 1 + (i % 2)
            for k in range(n_img):
                iid = f"{tid}_{k}"; rel = f"data_image/{ev}/1_9_2017/{iid}.jpg"
                os.makedirs(os.path.join(root, os.path.dirname(rel)), exist_ok=True)
                Image.fromarray((rng.rand(48, 64, 3) * 255).astype(np.uint8)).save(os.path.join(root, rel))
                rows.append(dict(event_name=ev, tweet_id=str(tid), image_id=iid, tweet_text=text, image_path=rel,
                                 hum=hum8[(i + k) % 8], inf=['informative', 'not_informative'][(i + k) % 2],
                                 inf_t=['informative', 'not_informative'][i % 2], dmg=(i + k) % 3))
        df = pd.DataFrame(rows)
        for fam in ['informative', 'humanitarian', 'damage']:
            os.makedirs(os.path.join(root, fam), exist_ok=True)
            if fam == 'informative':      # string labels + a text-label column  -> path (a)
                out = df.assign(label=df['inf'], label_text=df['inf_t'])
            elif fam == 'humanitarian':   # string labels, no text labels       -> weak path (c)
                out = df.assign(label=df['hum'])
            else:                         # integer ClassLabel                  -> numeric path
                out = df.iloc[::2].assign(label=df['dmg'].iloc[::2])
            out[['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image_path', 'label'] +
                (['label_text'] if 'label_text' in out else [])].to_json(
                os.path.join(root, fam, f"{split}.json"), orient='records', lines=True)


def fetch(url, dst):
    if os.path.exists(dst): return dst
    print("downloading", url)
    urllib.request.urlretrieve(url, dst + ".part"); os.replace(dst + ".part", dst)
    return dst


def ensure_official_splits():
    if glob.glob(os.path.join(OFFICIAL_DIR, "**", "*.tsv"), recursive=True): return True
    if SMOKE: return False
    try:
        z = fetch(OFFICIAL_SPLIT_URL, os.path.join(DATA_ROOT, "crisismmd_datasplit_all.zip"))
        with zipfile.ZipFile(z) as zf: zf.extractall(OFFICIAL_DIR)
        return True
    except Exception as e:
        print("  official split download failed:", repr(e)); return False


def ensure_data():
    have_json = all(glob.glob(os.path.join(DATA_ROOT, f, "*.json*")) for f in FAMILIES)
    have_img = os.path.isdir(os.path.join(DATA_ROOT, "data_image"))
    if have_json and have_img:
        print("data already present"); return 'hf'
    if SMOKE:
        make_smoke_data(DATA_ROOT); return 'smoke'
    try:
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=DATA_ROOT, max_workers=8,
                          allow_patterns=[f"{f}/*" for f in FAMILIES] + ["data_image/**"])
        return 'hf'
    except Exception as e:
        print("Hugging Face download failed:", repr(e), "-> trying the official CrisisNLP files")
    assert ensure_official_splits(), "could not obtain CrisisMMD annotations from either source"
    tgz = fetch(OFFICIAL_DATA_URL, os.path.join(DATA_ROOT, "CrisisMMD_v2.0.tar.gz"))
    with tarfile.open(tgz) as tf: tf.extractall(DATA_ROOT)
    return 'official'

DATA_SOURCE = ensure_data()
print("data source:", DATA_SOURCE)
print(sorted(os.listdir(DATA_ROOT))[:20])

In [ ]:
def read_any_json(p):
    try:
        return pd.read_json(p, lines=True, dtype=False)
    except ValueError:
        pass
    with open(p) as fh:
        obj = json.load(fh)
    if isinstance(obj, dict):
        for k in ('data', 'rows', 'records', 'examples'):
            if isinstance(obj.get(k), list):
                obj = obj[k]; break
        else:
            obj = list(obj.values()) if all(isinstance(v, dict) for v in obj.values()) else [obj]
    return pd.DataFrame(obj)


def _norm(s):
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', str(s).lower())).strip('_')

# integer -> name order of the Hugging Face ClassLabel features (alphabetical, as shown in the HF viewer)
HF_INT_NAMES = {'damage': ['little_or_no_damage', 'mild_damage', 'severe_damage'],
                'humanitarian': ['affected_individuals', 'infrastructure_and_utility_damage', 'injured_or_dead_people',
                                 'missing_or_found_people', 'not_humanitarian', 'other_relevant_information',
                                 'rescue_volunteering_or_donation_effort', 'vehicle_damage'],
                'informative': ['informative', 'not_informative']}


def canon(family, raw):
    """raw label (string or HF integer) -> canonical class name, or None (e.g. 'dont_know')."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)): return None
    if isinstance(raw, (int, np.integer)) or (isinstance(raw, str) and raw.strip().isdigit()):
        raw = HF_INT_NAMES[family][int(raw)]
    s = _norm(raw)
    if family == 'damage':
        if 'severe' in s: return 'severe_damage'
        if 'mild' in s: return 'mild_damage'
        if 'little' in s or s in ('no_damage', 'none'): return 'little_or_no_damage'
        return None
    if family == 'informative':
        if s.startswith('not') or 'not_informative' in s: return 'not_informative'
        if s == 'informative': return 'informative'
        return None
    # humanitarian: standard 8 -> 5 merge (as in Ofli, Alam & Imran 2020)
    if 'not_humanitarian' in s or 'not_relevant' in s: return 'not_humanitarian'
    if 'infrastructure' in s or 'vehicle' in s: return 'infrastructure_and_utility_damage'
    if any(w in s for w in ('injured', 'dead', 'affected', 'missing', 'found_people')): return 'affected_injured_or_dead_people'
    if any(w in s for w in ('rescue', 'volunteer', 'donation')): return 'rescue_volunteering_or_donation_effort'
    if 'other' in s: return 'other_relevant_information'
    return None


def _pick(df, *names):
    for n in names:
        if n in df.columns: return n
    return None


def find_official_tsv(family, split):
    files = glob.glob(os.path.join(OFFICIAL_DIR, "**", "*.tsv"), recursive=True)
    cand = [f for f in files if f"task_{family}" in os.path.basename(f) and re.search(rf"_{split}\.tsv$", f)]
    cand.sort(key=lambda f: ('text_img' not in f, f))
    return cand


def read_tsv(p):
    return pd.read_csv(p, sep='\t', dtype=str, quoting=csv.QUOTE_NONE, on_bad_lines='skip')


def load_raw_table(family, split):
    files = sorted(glob.glob(os.path.join(DATA_ROOT, family, f"{split}*.json*")))
    if files:
        return read_any_json(files[0]), f"hf:{os.path.relpath(files[0], DATA_ROOT)}"
    ensure_official_splits()
    c = find_official_tsv(family, split)
    assert c, f"no annotation file for {family}/{split}"
    return read_tsv(c[0]), f"official:{os.path.basename(c[0])}"


IMG_INDEX = None
def resolve_image(p):
    global IMG_INDEX
    if isinstance(p, dict): p = p.get('path')
    if not isinstance(p, str) or not p: return None
    for cand in (p, os.path.join(DATA_ROOT, p), os.path.join(DATA_ROOT, "CrisisMMD_v2.0", p)):
        if os.path.isfile(cand): return cand
    if IMG_INDEX is None:
        IMG_INDEX = {os.path.basename(f): f for f in glob.glob(os.path.join(DATA_ROOT, "**", "*.*"), recursive=True)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))}
    return IMG_INDEX.get(os.path.basename(p))


TABLE_SOURCES = {}
def load_image_table(family, split):
    df, src = load_raw_table(family, split)
    TABLE_SOURCES[f"{family}/{split}"] = src
    ic = _pick(df, 'image_path', 'image', 'img_path')
    lc = _pick(df, 'label_image', 'image_label', 'label')
    tc = _pick(df, 'label_text', 'text_label')
    out = pd.DataFrame({
        'event_name': df[_pick(df, 'event_name', 'event')].astype(str),
        'tweet_id': df['tweet_id'].astype(str),
        'image_id': df[_pick(df, 'image_id')].astype(str) if _pick(df, 'image_id') else df[ic].astype(str),
        'tweet_text': df['tweet_text'].astype(str),
        'image_ref': df[ic],
        'label_raw': df[lc],
        'label_text_raw': df[tc] if tc else None,
    })
    out['split'] = split
    out['label_name'] = [canon(family, v) for v in out['label_raw']]
    out['label_text_name'] = [canon(family, v) for v in out['label_text_raw']] if tc else None
    return out

RAW = {fam: {s: load_image_table(fam, s) for s in SPLITS} for fam in FAMILIES}
for fam in FAMILIES:
    for s in SPLITS:
        t = RAW[fam][s]
        print(f"{fam:12s} {s:5s} rows {len(t):6d} | label values {dict(t['label_name'].value_counts(dropna=False))}")
print("\nsources:", json.dumps(TABLE_SOURCES, indent=1))

### 2b. Resolve image files, cache resized copies, and fingerprint them

Each image is opened once: corrupt files are dropped (counted), an aspect-preserving copy with a 256-px short side is cached as JPEG for fast training, and the MD5 of the original bytes is kept for duplicate detection across splits.

In [ ]:
all_refs = pd.concat([RAW[f][s][['image_id', 'image_ref']] for f in FAMILIES for s in SPLITS]).drop_duplicates('image_id')

def _prep_image(row):
    iid, ref = row
    src = resolve_image(ref)
    if src is None: return iid, None, None, 'missing'
    dst = os.path.join(CACHE_DIR, hashlib.md5(iid.encode()).hexdigest() + ".jpg")
    try:
        with open(src, 'rb') as fh: raw = fh.read()
        md5 = hashlib.md5(raw).hexdigest()
        if not os.path.exists(dst):
            im = Image.open(io.BytesIO(raw)).convert('RGB')
            w, h = im.size; s = 256 / min(w, h)
            im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR).save(dst, quality=92)
        return iid, dst, md5, 'ok'
    except Exception as e:
        return iid, None, None, f'corrupt'

with ThreadPoolExecutor(8) as ex:
    res = list(tqdm(ex.map(_prep_image, all_refs[['image_id', 'image_ref']].itertuples(index=False, name=None)),
                    total=len(all_refs), desc="caching images"))
IMG = pd.DataFrame(res, columns=['image_id', 'cache_path', 'md5', 'status']).set_index('image_id')
print(IMG['status'].value_counts().to_string())
for fam in FAMILIES:
    for s in SPLITS:
        t = RAW[fam][s]
        t['cache_path'] = IMG.loc[t['image_id'], 'cache_path'].values
        t['md5'] = IMG.loc[t['image_id'], 'md5'].values

---
## 3. Build the task tables + leakage control

* **Image tasks**: one row per image with a valid label and a readable file.
* **Text tasks**: one row per tweet (texts are repeated across a tweet's images, so they are de-duplicated by `tweet_id`).
* **Leakage**: CrisisMMD contains many retweets and re-posted images. TRAIN rows that share a tweet id, a normalised text (text tasks) or an identical image file (image tasks) with any dev/test row are dropped; DEV rows that duplicate TEST are dropped too. **The test split is never modified.** All counts are written to `data_report/leakage_report.json`.

In [ ]:
def norm_text(t):
    t = str(t).lower()
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'^rt\s+@\w+:?', ' ', t.strip())
    t = re.sub(r'@\w+', '@user', t)
    t = re.sub(r'&amp;', '&', t)
    return re.sub(r'\s+', ' ', t).strip()

def model_text(t):
    """light cleaning for the transformer: keep wording, mask URLs and user handles."""
    t = re.sub(r'https?://\S+', 'http', str(t))
    t = re.sub(r'@\w+', '@user', t)
    t = t.replace('&amp;', '&')
    return re.sub(r'\s+', ' ', t).strip()

LEAK = {}
TEXT_LABEL_SOURCE = {}

def build_image_task(task):
    spec = TASKS[task]; fam = spec['family']; cls = spec['classes']
    parts = {}
    for s in SPLITS:
        t = RAW[fam][s]
        t = t[t['label_name'].isin(cls) & t['cache_path'].notna()].drop_duplicates('image_id').copy()
        t['label'] = t['label_name'].map({c: i for i, c in enumerate(cls)}).astype(int)
        parts[s] = t
    rep = {'before': {s: len(parts[s]) for s in SPLITS}}
    assert not (set(parts['test']['image_id']) & set(parts['train']['image_id'])), "same image id in train and test"
    if DROP_TRAIN_LEAKS:
        for s, refs in [('train', ['dev', 'test']), ('dev', ['test'])]:
            bad_t = set().union(*[set(parts[r]['tweet_id']) for r in refs])
            bad_m = set().union(*[set(parts[r]['md5']) for r in refs])
            m = parts[s]['tweet_id'].isin(bad_t) | parts[s]['md5'].isin(bad_m)
            rep[f'{s}_dropped_shared_tweet_or_identical_image'] = int(m.sum())
            parts[s] = parts[s][~m]
    rep['after'] = {s: len(parts[s]) for s in SPLITS}
    LEAK[task] = rep
    return parts


def build_text_task(task):
    spec = TASKS[task]; fam = spec['family']; cls = spec['classes']
    parts, source = {}, None
    for s in SPLITS:
        t = RAW[fam][s]
        if t['label_text_name'] is not None and t['label_text_name'].notna().any():          # (a) text labels in source
            u = t.dropna(subset=['label_text_name']).drop_duplicates('tweet_id').assign(label_name_t=lambda x: x['label_text_name'])
            source = 'source label_text column'
        else:
            u = None
            for p in find_official_tsv(fam, s) if ensure_official_splits() else []:          # (b) official split files
                o = read_tsv(p)
                lc = _pick(o, 'label_text') or ('label' if ('_text_' in os.path.basename(p) and 'img' not in os.path.basename(p)) else None)
                if lc:
                    u = pd.DataFrame({'event_name': o['event_name'].astype(str), 'tweet_id': o['tweet_id'].astype(str),
                                      'tweet_text': o['tweet_text'].astype(str), 'label_name_t': [canon(fam, v) for v in o[lc]]})
                    u = u.dropna(subset=['label_name_t']).drop_duplicates('tweet_id')
                    source = f'official split file {os.path.basename(p)} ({lc})'
                    break
            if u is None:                                                                     # (c) weak fallback
                g = t.dropna(subset=['label_name']).groupby('tweet_id')
                agree = g['label_name'].nunique() == 1
                u = t[t['tweet_id'].isin(agree[agree].index)].drop_duplicates('tweet_id').assign(label_name_t=lambda x: x['label_name'])
                source = 'WEAK: derived from image labels (tweets whose images all agree)'
        u = u[u['label_name_t'].isin(cls)].copy()
        u['label'] = u['label_name_t'].map({c: i for i, c in enumerate(cls)}).astype(int)
        u['ntext'] = u['tweet_text'].map(norm_text)
        u['split'] = s
        parts[s] = u[['event_name', 'tweet_id', 'tweet_text', 'ntext', 'label', 'split']].reset_index(drop=True)
    TEXT_LABEL_SOURCE[task] = source
    if source.startswith('WEAK'):
        print(f"!! {task}: text labels are WEAK ({source}). Say so in the paper, or fix the official split download.")
    rep = {'label_source': source, 'before': {s: len(parts[s]) for s in SPLITS}}
    if DROP_TRAIN_LEAKS:
        for s, refs in [('train', ['dev', 'test']), ('dev', ['test'])]:
            bad_id = set().union(*[set(parts[r]['tweet_id']) for r in refs])
            bad_tx = set().union(*[set(parts[r]['ntext']) for r in refs])
            m = parts[s]['tweet_id'].isin(bad_id) | parts[s]['ntext'].isin(bad_tx)
            rep[f'{s}_dropped_shared_tweet_or_duplicate_text'] = int(m.sum())
            parts[s] = parts[s][~m].reset_index(drop=True)
    rep['after'] = {s: len(parts[s]) for s in SPLITS}
    LEAK[task] = rep
    return parts


DATA = {}
for task, spec in TASKS.items():
    DATA[task] = (build_text_task if spec['modality'] == 'text' else build_image_task)(task)

rows = []
for task, spec in TASKS.items():
    for s in SPLITS:
        vc = DATA[task][s]['label'].value_counts()
        rows.append({'task': task, 'split': s, 'n': len(DATA[task][s]),
                     **{c: int(vc.get(i, 0)) for i, c in enumerate(spec['classes'])}})
dist = pd.DataFrame(rows).fillna(0)
dist.to_csv(os.path.join(D['data'], "class_distribution.csv"), index=False)
jdump({'leakage': LEAK, 'text_label_source': TEXT_LABEL_SOURCE, 'table_sources': TABLE_SOURCES,
       'image_status': IMG['status'].value_counts().to_dict()}, os.path.join(D['data'], "leakage_report.json"))
pd.set_option('display.width', 220)
print(dist.to_string(index=False))
print("\nleakage control:\n" + json.dumps(LEAK, indent=1))

for task in TASKS:
    tr, te = DATA[task]['train'], DATA[task]['test']
    assert not (set(tr['tweet_id']) & set(te['tweet_id'])) or not DROP_TRAIN_LEAKS, f"{task}: tweet overlap train/test"
print("\n[OK] no train/test tweet overlap in any task" if DROP_TRAIN_LEAKS else "\n(leak dropping disabled)")

---
## 4. Metrics

Accuracy, macro and weighted precision/recall/F1, MCC, Cohen's κ, one-vs-rest ROC-AUC, and quadratic-weighted κ for the ordinal damage task. Every plot is saved with its numbers.

In [ ]:
def compute_metrics(y, pred, prob, n_classes, ordinal=False):
    m = {'accuracy': accuracy_score(y, pred)}
    for avg in ['macro', 'weighted']:
        m[f'precision_{avg}'] = precision_score(y, pred, average=avg, zero_division=0)
        m[f'recall_{avg}'] = recall_score(y, pred, average=avg, zero_division=0)
        m[f'f1_{avg}'] = f1_score(y, pred, average=avg, zero_division=0)
    m['mcc'] = matthews_corrcoef(y, pred)
    m['cohen_kappa'] = cohen_kappa_score(y, pred)
    try:
        m['roc_auc_macro'] = (roc_auc_score(y, prob[:, 1]) if n_classes == 2 else
                              roc_auc_score(y, prob, multi_class='ovr', average='macro', labels=list(range(n_classes))))
    except Exception:
        m['roc_auc_macro'] = float('nan')
    if ordinal:
        m['qwk'] = cohen_kappa_score(y, pred, weights='quadratic')
    return {k: float(v) for k, v in m.items()}


def save_report(y, pred, prob, names, title, prefix, ordinal=False):
    m = compute_metrics(y, pred, prob, len(names), ordinal)
    rep = classification_report(y, pred, labels=list(range(len(names))), target_names=names, digits=4, zero_division=0)
    with open(prefix + "_classification_report.txt", "w") as fh: fh.write(rep)
    cm = confusion_matrix(y, pred, labels=list(range(len(names))))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    pd.DataFrame(cm, index=names, columns=names).to_csv(prefix + "_confusion_matrix.csv")
    pd.DataFrame(cmn, index=names, columns=names).to_csv(prefix + "_confusion_matrix_normalized.csv")
    short = [n.replace('_', ' ')[:22] for n in names]
    fig, ax = plt.subplots(1, 2, figsize=(5 + 2.4 * len(names), 4.2))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0], xticklabels=short, yticklabels=short, cbar=False)
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', ax=ax[1], xticklabels=short, yticklabels=short, cbar=False, vmin=0, vmax=1)
    for a, t in zip(ax, ['counts', 'row-normalised (recall)']):
        a.set_title(f"{title} - {t}", fontsize=9); a.set_xlabel('predicted'); a.set_ylabel('actual')
        a.tick_params(labelsize=7)
    plt.tight_layout(); plt.savefig(prefix + "_confusion_matrix.png", dpi=150, bbox_inches='tight'); plt.close(fig)
    jdump(m, prefix + "_metrics.json")
    return m

print("metrics ready")

---
## 5. Text baselines — majority class and TF-IDF + logistic regression

The regularisation strength is chosen on **dev** macro-F1; test is scored once.

In [ ]:
RESULTS = []          # one row per (task, model, seed) on the TEST split

for task, spec in TASKS.items():
    if spec['modality'] != 'text': continue
    tr, dv, te = DATA[task]['train'], DATA[task]['dev'], DATA[task]['test']
    n = len(spec['classes'])
    maj = int(tr['label'].value_counts().idxmax())
    prob = np.zeros((len(te), n)); prob[:, maj] = 1
    m = compute_metrics(te['label'].values, np.full(len(te), maj), prob, n)
    RESULTS.append({'task': task, 'model': 'majority class', 'seed': None, **m})

    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=50000)
    Xtr, Xdv, Xte = vec.fit_transform(tr['ntext']), vec.transform(dv['ntext']), vec.transform(te['ntext'])
    best = None
    for C in [0.25, 1.0, 4.0]:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight='balanced').fit(Xtr, tr['label'])
        f = f1_score(dv['label'], clf.predict(Xdv), average='macro')
        if best is None or f > best[0]: best = (f, C, clf)
    clf = best[2]
    pr = clf.predict_proba(Xte)
    full = np.zeros((len(te), n)); full[:, clf.classes_] = pr
    pfx = os.path.join(D['metrics'], f"{task}__tfidf_lr")
    m = save_report(te['label'].values, full.argmax(1), full, spec['classes'], f"{task} TF-IDF+LR", pfx)
    RESULTS.append({'task': task, 'model': 'TF-IDF + LogReg', 'seed': None, 'dev_macro_f1': best[0], 'C': best[1], **m})
    print(f"{task:18s} majority F1 {RESULTS[-2]['f1_macro']:.4f} | TF-IDF+LR (C={best[1]}) test macro-F1 {m['f1_macro']:.4f}")

---
## 6. Transformer text classifiers

Fine-tuned `distilroberta-base` (configurable), class-weighted cross-entropy, linear warm-up/decay, model selection on **dev macro-F1** with early stopping, then one test evaluation per seed. The best checkpoint of every seed is kept; the seed with the best **dev** score (never test) becomes the deployment model.

In [ ]:
def build_smoke_tokenizer(texts, path):
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers
    from transformers import PreTrainedTokenizerFast
    tk = Tokenizer(models.WordLevel(unk_token="<unk>"))
    tk.pre_tokenizer = pre_tokenizers.Whitespace()
    tk.train_from_iterator(texts, trainers.WordLevelTrainer(special_tokens=["<pad>", "<unk>", "<s>", "</s>"]))
    tok = PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="<unk>", pad_token="<pad>", cls_token="<s>", sep_token="</s>")
    tok.save_pretrained(path)
    return tok


def load_text_backbone(n_classes, texts=None, tok_dir=None):
    if SMOKE:
        from transformers import RobertaConfig, RobertaForSequenceClassification
        tok = build_smoke_tokenizer(texts, tok_dir)
        cfg = RobertaConfig(vocab_size=len(tok), hidden_size=32, num_hidden_layers=1, num_attention_heads=2,
                            intermediate_size=64, max_position_embeddings=TEXT_MAXLEN + 8, num_labels=n_classes,
                            pad_token_id=tok.pad_token_id, bos_token_id=tok.cls_token_id, eos_token_id=tok.sep_token_id)
        return tok, RobertaForSequenceClassification(cfg)
    tok = AutoTokenizer.from_pretrained(TEXT_MODEL)
    return tok, AutoModelForSequenceClassification.from_pretrained(TEXT_MODEL, num_labels=n_classes)


class TextDS(Dataset):
    def __init__(self, df): self.t = [model_text(x) for x in df['tweet_text']]; self.y = df['label'].tolist()
    def __len__(self): return len(self.t)
    def __getitem__(self, i): return self.t[i], self.y[i]


def text_collate(tok):
    def f(batch):
        t, y = zip(*batch)
        enc = tok(list(t), truncation=True, max_length=TEXT_MAXLEN, padding=True, return_tensors='pt')
        enc.pop('token_type_ids', None)
        enc['labels'] = torch.tensor(y, dtype=torch.long)
        return enc
    return f


@torch.no_grad()
def text_predict(model, loader):
    model.eval(); P = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != 'labels'}
        with torch.amp.autocast('cuda', enabled=AMP):
            P.append(F.softmax(model(**b).logits.float(), 1).cpu().numpy())
    return np.concatenate(P)


def train_text(task, seed):
    spec = TASKS[task]; cls = spec['classes']; n = len(cls)
    run = os.path.join(D['ckpt'], 'text', task, f"seed_{seed}")
    done_p = os.path.join(run, "done.json")
    if FORCE_RETRAIN: shutil.rmtree(run, ignore_errors=True)
    if RESUME and os.path.exists(done_p):
        print(f"  [RESUME] {task} seed {seed} already complete - loaded"); return json.load(open(done_p))
    os.makedirs(run, exist_ok=True)
    set_seed(seed)
    tr, dv, te = DATA[task]['train'], DATA[task]['dev'], DATA[task]['test']
    tok, model = load_text_backbone(n, texts=[model_text(x) for x in tr['tweet_text']], tok_dir=os.path.join(run, 'tok'))
    model.to(device)
    col = text_collate(tok)
    g = torch.Generator(); g.manual_seed(seed)
    tl = DataLoader(TextDS(tr), batch_size=TEXT_BATCH, shuffle=True, collate_fn=col, generator=g)
    dl = DataLoader(TextDS(dv), batch_size=TEXT_BATCH * 2, collate_fn=col)
    el = DataLoader(TextDS(te), batch_size=TEXT_BATCH * 2, collate_fn=col)
    w = class_weights(tr['label'].values, n)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=TEXT_LR, weight_decay=0.01)
    steps = len(tl) * TEXT_EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(TEXT_WARMUP * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    best_f1, best_ep, bad, hist = -1, 0, 0, []
    best_dir = os.path.join(run, 'best')
    for ep in range(1, TEXT_EPOCHS + 1):
        model.train(); t0 = time.time(); tot = 0
        for b in tqdm(tl, desc=f"{task} s{seed} e{ep}", leave=False):
            b = {k: v.to(device) for k, v in b.items()}
            y = b.pop('labels')
            with torch.amp.autocast('cuda', enabled=AMP):
                loss = crit(model(**b).logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            tot += loss.item() * len(y)
        pdv = text_predict(model, dl)
        f = f1_score(dv['label'], pdv.argmax(1), average='macro')
        hist.append({'epoch': ep, 'train_loss': tot / len(tr), 'dev_macro_f1': float(f),
                     'dev_acc': float(accuracy_score(dv['label'], pdv.argmax(1)))})
        mark = ""
        if f > best_f1:
            best_f1, best_ep, bad = f, ep, 0
            model.save_pretrained(best_dir); tok.save_pretrained(best_dir); mark = "  <- best (saved)"
        else:
            bad += 1
        print(f"  {task} seed {seed} epoch {ep}/{TEXT_EPOCHS} | loss {tot/len(tr):.4f} | dev macro-F1 {f:.4f} | {time.time()-t0:.0f}s{mark}")
        if bad >= TEXT_PATIENCE: print("  early stopping"); break
    del model; gc.collect(); torch.cuda.empty_cache()
    model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(device)
    pdv, pte = text_predict(model, dl), text_predict(model, el)
    np.savez(os.path.join(run, "outputs.npz"), dev_prob=pdv, test_prob=pte)
    pfx = os.path.join(D['metrics'], f"{task}__transformer_seed{seed}")
    m = save_report(te['label'].values, pte.argmax(1), pte, cls, f"{task} seed {seed}", pfx)
    pd.DataFrame(hist).to_csv(os.path.join(run, "history.csv"), index=False)
    out = {'task': task, 'model': f'transformer ({TEXT_MODEL})', 'seed': seed, 'best_epoch': best_ep,
           'dev_macro_f1': float(best_f1), 'best_dir': best_dir, 'history': hist, 'test': m}
    jdump(out, done_p)
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

TEXT_RUNS = {}
for task, spec in TASKS.items():
    if spec['modality'] != 'text': continue
    TEXT_RUNS[task] = [train_text(task, s) for s in TEXT_SEEDS]
    for r in TEXT_RUNS[task]:
        RESULTS.append({'task': task, 'model': r['model'], 'seed': r['seed'], 'dev_macro_f1': r['dev_macro_f1'],
                        'best_epoch': r['best_epoch'], **r['test']})
    print(f"{task}: test macro-F1 per seed {[round(r['test']['f1_macro'], 4) for r in TEXT_RUNS[task]]}")

---
## 7. Image classifiers — ConvNeXt-Tiny (ImageNet-pretrained)

Same backbone family as the satellite model. Train: random resized crop, flips, mild colour jitter. Dev/test: resize 256 → centre crop 224. Class-weighted CE with label smoothing, cosine LR, early stopping on dev macro-F1. Resumable per epoch.

In [ ]:
NORM_MEAN, NORM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
img_train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2), transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])
img_eval_tf = transforms.Compose([
    transforms.Resize(round(IMG_SIZE * 256 / 224)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(NORM_MEAN, NORM_STD)])


class ImgDS(Dataset):
    def __init__(self, df, tf): self.p = df['cache_path'].tolist(); self.y = df['label'].tolist(); self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]


def make_img_model(n, pretrained=None):
    return timm.create_model(IMG_MODEL, pretrained=(not SMOKE) if pretrained is None else pretrained, num_classes=n)


@torch.no_grad()
def img_predict(model, loader):
    model.eval(); P = []
    for x, _ in loader:
        with torch.amp.autocast('cuda', enabled=AMP):
            P.append(F.softmax(model(x.to(device)).float(), 1).cpu().numpy())
    return np.concatenate(P)


def train_image(task, seed):
    spec = TASKS[task]; cls = spec['classes']; n = len(cls)
    run = os.path.join(D['ckpt'], 'image', task, f"seed_{seed}")
    done_p, last_p, best_p = (os.path.join(run, f) for f in ("done.json", "last.pth", "best.pth"))
    if FORCE_RETRAIN: shutil.rmtree(run, ignore_errors=True)
    if RESUME and os.path.exists(done_p):
        print(f"  [RESUME] {task} seed {seed} already complete - loaded"); return json.load(open(done_p))
    os.makedirs(run, exist_ok=True)
    set_seed(seed)
    tr, dv, te = DATA[task]['train'], DATA[task]['dev'], DATA[task]['test']
    g = torch.Generator(); g.manual_seed(seed)
    tl = DataLoader(ImgDS(tr, img_train_tf), batch_size=IMG_BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=True, drop_last=len(tr) > IMG_BATCH, generator=g)
    dl = DataLoader(ImgDS(dv, img_eval_tf), batch_size=IMG_BATCH * 2, num_workers=NUM_WORKERS)
    el = DataLoader(ImgDS(te, img_eval_tf), batch_size=IMG_BATCH * 2, num_workers=NUM_WORKERS)
    model = make_img_model(n).to(device)
    w = class_weights(tr['label'].values, n)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device), label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(model.parameters(), lr=IMG_LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=IMG_EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    best_f1, best_ep, bad, hist, start = -1.0, 0, 0, [], 1
    if RESUME and os.path.exists(last_p):
        ck = tload(last_p)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        scaler.load_state_dict(ck['scaler'])
        best_f1, best_ep, bad, hist, start = ck['best_f1'], ck['best_ep'], ck['bad'], ck['hist'], ck['epoch'] + 1
        print(f"  [RESUME] {task} seed {seed} continues at epoch {start}"); del ck
    meta = {'timm_model': IMG_MODEL, 'num_classes': n, 'class_names': cls, 'img_size': IMG_SIZE,
            'normalization': {'mean': NORM_MEAN, 'std': NORM_STD}, 'eval_resize': round(IMG_SIZE * 256 / 224),
            'task': task, 'seed': seed}
    for ep in range(start, IMG_EPOCHS + 1):
        if bad >= IMG_PATIENCE: break
        model.train(); t0 = time.time(); tot, nseen = 0.0, 0
        for x, y in tqdm(tl, desc=f"{task} s{seed} e{ep}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                loss = crit(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); nseen += len(y)
        sch.step()
        pdv = img_predict(model, dl)
        f = f1_score(dv['label'], pdv.argmax(1), average='macro')
        hist.append({'epoch': ep, 'train_loss': tot / max(nseen, 1), 'dev_macro_f1': float(f),
                     'dev_acc': float(accuracy_score(dv['label'], pdv.argmax(1)))})
        mark = ""
        if f > best_f1:
            best_f1, best_ep, bad = f, ep, 0
            atomic_torch_save({'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
                               'meta': {**meta, 'epoch': ep, 'dev_macro_f1': float(f)}}, best_p)
            mark = "  <- best (saved)"
        else:
            bad += 1
        atomic_torch_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sch': sch.state_dict(),
                           'scaler': scaler.state_dict(), 'epoch': ep, 'best_f1': best_f1, 'best_ep': best_ep,
                           'bad': bad, 'hist': hist}, last_p)
        print(f"  {task} seed {seed} epoch {ep}/{IMG_EPOCHS} | loss {tot/max(nseen,1):.4f} | dev macro-F1 {f:.4f} | {time.time()-t0:.0f}s{mark}")
        if bad >= IMG_PATIENCE: print("  early stopping")
    model.load_state_dict(tload(best_p)['state_dict'])
    pdv, pte = img_predict(model, dl), img_predict(model, el)
    np.savez(os.path.join(run, "outputs.npz"), dev_prob=pdv, test_prob=pte)
    pfx = os.path.join(D['metrics'], f"{task}__convnext_seed{seed}")
    m = save_report(te['label'].values, pte.argmax(1), pte, cls, f"{task} seed {seed}", pfx, ordinal=spec['ordinal'])
    pd.DataFrame(hist).to_csv(os.path.join(run, "history.csv"), index=False)
    if os.path.exists(last_p): os.remove(last_p)      # optimizer state no longer needed
    out = {'task': task, 'model': f'image ({IMG_MODEL})', 'seed': seed, 'best_epoch': best_ep,
           'dev_macro_f1': float(best_f1), 'best_path': best_p, 'history': hist, 'test': m}
    jdump(out, done_p)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return out

IMG_RUNS = {}
for task, spec in TASKS.items():
    if spec['modality'] != 'image': continue
    IMG_RUNS[task] = [train_image(task, s) for s in IMG_SEEDS]
    for r in IMG_RUNS[task]:
        RESULTS.append({'task': task, 'model': r['model'], 'seed': r['seed'], 'dev_macro_f1': r['dev_macro_f1'],
                        'best_epoch': r['best_epoch'], **r['test']})
    print(f"{task}: test macro-F1 per seed {[round(r['test']['f1_macro'], 4) for r in IMG_RUNS[task]]}")

---
## 8. Results tables (TEST split)

Per-seed rows and a mean ± SD summary across seeds. These are the numbers for the paper. Note: SD over seeds measures training randomness on a *fixed* split, not split-to-split variation.

In [ ]:
res = pd.DataFrame(RESULTS)
res.to_csv(os.path.join(D['metrics'], "test_results_per_seed.csv"), index=False)
KEYS = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'f1_weighted', 'mcc', 'cohen_kappa', 'roc_auc_macro', 'qwk']
summ = []
for (task, model), g in res.groupby(['task', 'model'], sort=False):
    row = {'task': task, 'model': model, 'n_seeds': int(g['seed'].notna().sum()) or 1}
    for k in KEYS:
        if k in g and g[k].notna().any():
            v = g[k].dropna().values
            row[k] = f"{v.mean():.4f} ± {v.std(ddof=1):.4f}" if len(v) > 1 else f"{v.mean():.4f}"
    summ.append(row)
summ = pd.DataFrame(summ)
summ.to_csv(os.path.join(D['metrics'], "test_results_summary.csv"), index=False)
print(summ.to_string(index=False))

# deployment model per task = best DEV macro-F1 (test never used for selection)
DEPLOY = {}
for task, runs in {**TEXT_RUNS, **IMG_RUNS}.items():
    r = max(runs, key=lambda r: r['dev_macro_f1'])
    DEPLOY[task] = r
    print(f"deploy {task:18s}: seed {r['seed']} (dev macro-F1 {r['dev_macro_f1']:.4f}, test macro-F1 {r['test']['f1_macro']:.4f})")
jdump({t: {k: v for k, v in r.items() if k != 'history'} for t, r in DEPLOY.items()},
      os.path.join(D['metrics'], "deployment_selection.json"))

---
## 9. Per-event breakdown and test predictions

How performance varies by disaster event (the deployment model of each task). Large drops on some events are a finding for the discussion section, not something to hide.

In [ ]:
ev_rows = []
for task, r in DEPLOY.items():
    spec = TASKS[task]; te = DATA[task]['test'].copy()
    run = os.path.dirname(r['best_dir'] if 'best_dir' in r else r['best_path'])
    prob = np.load(os.path.join(run, "outputs.npz"))['test_prob']
    te['pred'] = prob.argmax(1); te['confidence'] = prob.max(1)
    for i, c in enumerate(spec['classes']): te[f'prob_{c}'] = prob[:, i]
    te['ground_truth'] = [spec['classes'][i] for i in te['label']]
    te['prediction'] = [spec['classes'][i] for i in te['pred']]
    keep = ['event_name', 'tweet_id'] + (['image_id'] if 'image_id' in te else []) + ['tweet_text', 'ground_truth',
            'prediction', 'confidence'] + [f'prob_{c}' for c in spec['classes']]
    te[keep].to_csv(os.path.join(D['preds'], f"{task}_test_predictions.csv"), index=False)
    for ev, g in te.groupby('event_name'):
        ev_rows.append({'task': task, 'event': ev, 'n': len(g), 'accuracy': accuracy_score(g['label'], g['pred']),
                        'macro_f1': f1_score(g['label'], g['pred'], average='macro', zero_division=0)})
ev = pd.DataFrame(ev_rows).round(4)
ev.to_csv(os.path.join(D['metrics'], "test_results_by_event.csv"), index=False)
print(ev.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
piv = ev.pivot(index='event', columns='task', values='macro_f1')
piv.plot.bar(ax=ax); ax.set_ylabel('test macro-F1'); ax.set_ylim(0, 1); ax.grid(alpha=.3, axis='y')
ax.set_title('Per-event test macro-F1 (deployment model per task)'); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig(os.path.join(D['plots'], "per_event_macro_f1.png"), dpi=150); plt.close(fig)

---
## 10. Inference bundle — the code the demo will run

`news_bundle/news_inference.py` exposes `load()` and `analyze_item(text=None, image=None)`. The notebook imports this exact file for the case study, so the paper's case study and the demo run identical code.

* Text → informative probability, humanitarian category, **locations** (pretrained NER on the tweet with hashtags split, e.g. `#HoustonStrong` → "Houston Strong"), hazard keywords (rule-based).
* Image → informative probability, humanitarian category, damage severity with expected severity `E = 0.5·P(mild) + 1·P(severe)`.
* `priority` is a **transparent heuristic**, not a learned or evaluated quantity: `relevance × impact`, where relevance is the mean available P(informative) and impact is the max of the image's expected damage severity and a fixed weight for the predicted text category (injured/dead 1.0, infrastructure 0.8, rescue 0.5, other 0.3, not humanitarian 0).

In [ ]:
B = D['bundle']
os.makedirs(os.path.join(B, 'models'), exist_ok=True)
for task, r in DEPLOY.items():
    dst = os.path.join(B, 'models', task)
    shutil.rmtree(dst, ignore_errors=True)
    if TASKS[task]['modality'] == 'text':
        shutil.copytree(r['best_dir'], dst)
    else:
        os.makedirs(dst, exist_ok=True); shutil.copy2(r['best_path'], os.path.join(dst, 'model.pth'))
bundle_cfg = {'tasks': {t: {'modality': TASKS[t]['modality'], 'classes': TASKS[t]['classes'],
                            'ordinal': TASKS[t]['ordinal'], 'path': f"models/{t}", 'seed': DEPLOY[t]['seed'],
                            'dev_macro_f1': DEPLOY[t]['dev_macro_f1'], 'test_macro_f1': DEPLOY[t]['test']['f1_macro']}
                        for t in DEPLOY},
              'text_maxlen': TEXT_MAXLEN, 'img_size': IMG_SIZE, 'eval_resize': round(IMG_SIZE * 256 / 224),
              'normalization': {'mean': NORM_MEAN, 'std': NORM_STD}, 'ner_model': None if SMOKE else NER_MODEL,
              'priority_rule': 'relevance x impact (heuristic, not evaluated)',
              'text_category_weight': {'affected_injured_or_dead_people': 1.0, 'infrastructure_and_utility_damage': 0.8,
                                       'rescue_volunteering_or_donation_effort': 0.5, 'other_relevant_information': 0.3,
                                       'not_humanitarian': 0.0},
              'created': datetime.datetime.now().isoformat(timespec='seconds')}
jdump(bundle_cfg, os.path.join(B, 'config.json'))

NEWS_INFERENCE_PY = r'''"""
Disaster XAI-Net - news / social-media module: inference API (CrisisMMD-trained).

    from news_inference import load, analyze_item
    load()                                         # once
    analyze_item(text="Roof torn off in Rockport #HurricaneHarvey", image="photo.jpg")

`image` may be a path, PIL image, HxWx3 uint8 array or raw bytes. Either input may be omitted.
These models are for ground-level news/social-media content. They are NOT the satellite
damage model and must not be given overhead pre/post tiles.
"""
import io, os, re, json, time, urllib.parse, urllib.request
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

BUNDLE_DIR = os.path.dirname(os.path.abspath(__file__))
_S = {}

HAZARD_KEYWORDS = {   # rule-based, matches the satellite model's hazard names
    'cyclone':    ['hurricane', 'cyclone', 'typhoon', 'tropical storm', 'storm surge'],
    'earthquake': ['earthquake', 'quake', 'tremor', 'aftershock', 'seismic', 'magnitude'],
    'flood':      ['flood', 'flooding', 'flooded', 'inundat', 'waterlogged', 'deluge'],
    'wildfire':   ['wildfire', 'bushfire', 'forest fire', 'blaze', 'fire', 'burned', 'burnt', 'smoke'],
}


def load(bundle_dir=None, device=None, ner=True):
    bundle_dir = bundle_dir or BUNDLE_DIR
    if _S.get('dir') == bundle_dir: return _S
    import timm
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    cfg = json.load(open(os.path.join(bundle_dir, 'config.json')))
    models = {}
    for t, c in cfg['tasks'].items():
        p = os.path.join(bundle_dir, c['path'])
        if c['modality'] == 'text':
            models[t] = (AutoTokenizer.from_pretrained(p), AutoModelForSequenceClassification.from_pretrained(p).to(dev).eval())
        else:
            ck = torch.load(os.path.join(p, 'model.pth'), map_location='cpu', weights_only=False)
            m = timm.create_model(ck['meta']['timm_model'], pretrained=False, num_classes=ck['meta']['num_classes'])
            m.load_state_dict(ck['state_dict']); models[t] = m.to(dev).eval()
    ner_pipe = None
    if ner and cfg.get('ner_model'):
        try:
            from transformers import pipeline
            ner_pipe = pipeline('token-classification', model=cfg['ner_model'], aggregation_strategy='simple',
                                device=0 if dev.type == 'cuda' else -1)
        except Exception as e:
            print('NER model unavailable, using the rule-based fallback:', repr(e))
    _S.update(dir=bundle_dir, cfg=cfg, dev=dev, models=models, ner=ner_pipe, geocache={})
    return _S


def _to_pil(x):
    if isinstance(x, Image.Image): return x.convert('RGB')
    if isinstance(x, (bytes, bytearray)): return Image.open(io.BytesIO(x)).convert('RGB')
    if isinstance(x, (str, os.PathLike)): return Image.open(x).convert('RGB')
    if isinstance(x, np.ndarray): return Image.fromarray(x.astype(np.uint8)[..., :3]).convert('RGB')
    raise TypeError(f'unsupported image type {type(x)}')


def _img_tensor(im):
    c = _S['cfg']; r, s = c['eval_resize'], c['img_size']
    w, h = im.size; k = r / min(w, h)
    im = im.resize((max(1, round(w * k)), max(1, round(h * k))), Image.BILINEAR)
    w, h = im.size; l, t = (w - s) // 2, (h - s) // 2
    a = np.asarray(im.crop((l, t, l + s, t + s)), dtype=np.float32) / 255.0
    a = (a - np.array(c['normalization']['mean'])) / np.array(c['normalization']['std'])
    return torch.from_numpy(a.transpose(2, 0, 1)).float()[None]


def clean_text(t):
    t = re.sub(r'https?://\S+', 'http', str(t)); t = re.sub(r'@\w+', '@user', t)
    return re.sub(r'\s+', ' ', t.replace('&amp;', '&')).strip()


def split_hashtags(t):
    return re.sub(r'#(\w+)', lambda m: ' ' + re.sub(r'(?<=[a-z])(?=[A-Z])|_', ' ', m.group(1)) + ' ', str(t))


_STOP_LOC = {'rt', 'http', 'breaking', 'news', 'update', 'hurricane', 'earthquake', 'storm', 'fire', 'flood'}
def extract_locations(text):
    t = re.sub(r'https?://\S+', ' ', split_hashtags(text))
    t = re.sub(r'@\w+', ' ', t)
    locs = []
    if _S.get('ner') is not None:
        for e in _S['ner'](t):
            if e.get('entity_group') == 'LOC' and e.get('score', 0) >= 0.6:
                w = e['word'].replace(' ##', '').replace('##', '').strip(" .,:;'\"")
                if len(w) > 1: locs.append(w)
    else:   # fallback: capitalised phrase after a locative preposition
        for m in re.finditer(r'\b(?:in|at|near|from|across|of|hits?)\s+((?:[A-Z][a-zA-Z]+\s?){1,3})', t):
            locs.append(m.group(1).strip())
    seen, out = set(), []
    for l in locs:
        k = l.lower()
        if k not in seen and k not in _STOP_LOC:
            seen.add(k); out.append(l)
    return out


def hazard_keywords(text):
    t = str(text).lower()
    hits = {h: sum(t.count(k) for k in ks) for h, ks in HAZARD_KEYWORDS.items()}
    hits = {h: n for h, n in hits.items() if n}
    return {'hazards_mentioned': sorted(hits, key=hits.get, reverse=True), 'counts': hits, 'method': 'keyword rules'}


def geocode(name, sleep=1.1):
    """OpenStreetMap Nominatim lookup (cached; respect the 1 request/second policy)."""
    gc = _S.setdefault('geocache', {})
    if name in gc: return gc[name]
    try:
        url = 'https://nominatim.openstreetmap.org/search?' + urllib.parse.urlencode({'q': name, 'format': 'json', 'limit': 1})
        req = urllib.request.Request(url, headers={'User-Agent': 'disaster-xai-net-research/1.0'})
        r = json.loads(urllib.request.urlopen(req, timeout=10).read().decode())
        gc[name] = {'lat': float(r[0]['lat']), 'lon': float(r[0]['lon']), 'display_name': r[0]['display_name']} if r else None
        time.sleep(sleep)
    except Exception:
        gc[name] = None
    return gc[name]


@torch.no_grad()
def _text_probs(task, text):
    tok, m = _S['models'][task]
    enc = tok([clean_text(text)], truncation=True, max_length=_S['cfg']['text_maxlen'], return_tensors='pt')
    enc.pop('token_type_ids', None)
    return F.softmax(m(**{k: v.to(_S['dev']) for k, v in enc.items()}).logits.float(), 1)[0].cpu().numpy()


@torch.no_grad()
def _img_probs(task, x):
    return F.softmax(_S['models'][task](x.to(_S['dev'])).float(), 1)[0].cpu().numpy()


def _rec(task, p):
    names = _S['cfg']['tasks'][task]['classes']; k = int(np.argmax(p))
    return {'label': names[k], 'confidence': float(p[k]), 'probabilities': {n: float(p[i]) for i, n in enumerate(names)}}


def analyze_item(text=None, image=None, item_id=None, timestamp=None, do_geocode=False):
    S = load(); tasks = S['cfg']['tasks']; out = {'item_id': item_id, 'timestamp': timestamp, 'text': text}
    rel, impact = [], []
    if text:
        if 'text_informative' in tasks:
            r = _rec('text_informative', _text_probs('text_informative', text)); out['text_informative'] = r
            rel.append(r['probabilities']['informative'])
        if 'text_humanitarian' in tasks:
            r = _rec('text_humanitarian', _text_probs('text_humanitarian', text)); out['text_humanitarian'] = r
            impact.append(S['cfg']['text_category_weight'].get(r['label'], 0.0))
        out['locations'] = [{'name': l, **({'geo': geocode(l)} if do_geocode else {})} for l in extract_locations(text)]
        out['hazard'] = hazard_keywords(text)
    if image is not None:
        x = _img_tensor(_to_pil(image))
        if 'image_informative' in tasks:
            r = _rec('image_informative', _img_probs('image_informative', x)); out['image_informative'] = r
            rel.append(r['probabilities']['informative'])
        if 'image_humanitarian' in tasks:
            out['image_humanitarian'] = _rec('image_humanitarian', _img_probs('image_humanitarian', x))
        if 'image_damage' in tasks:
            p = _img_probs('image_damage', x); r = _rec('image_damage', p)
            r['expected_severity'] = float(0.5 * p[1] + 1.0 * p[2]); out['image_damage'] = r
            impact.append(r['expected_severity'])
    relevance = float(np.mean(rel)) if rel else 1.0
    out['priority'] = {'score': round(relevance * (max(impact) if impact else 0.0), 4),
                       'relevance': round(relevance, 4), 'impact': round(max(impact) if impact else 0.0, 4),
                       'method': 'heuristic relevance x impact (not a learned or evaluated quantity)'}
    return out


if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser(); ap.add_argument('--text'); ap.add_argument('--image')
    a = ap.parse_args()
    print(json.dumps(analyze_item(a.text, a.image), indent=2))
'''
with open(os.path.join(B, 'news_inference.py'), 'w') as fh:
    fh.write(NEWS_INFERENCE_PY)
with open(os.path.join(B, 'requirements.txt'), 'w') as fh:
    fh.write(f"torch>={torch.__version__.split('+')[0]}\ntransformers>={transformers.__version__}\ntimm=={timm.__version__}\nnumpy\npillow\n")
with open(os.path.join(B, 'README.md'), 'w') as fh:
    fh.write("# News / social-media module bundle\n\n`from news_inference import load, analyze_item`\n\n"
             "Models trained on CrisisMMD v2.0 (CC BY-NC-SA 4.0). Ground-level content only - not for satellite tiles.\n"
             "`priority` is a heuristic and was not evaluated. Location extraction uses a pretrained NER model and was "
             "not quantitatively evaluated except via the manual check sheet.\n")

spec_ = importlib.util.spec_from_file_location("news_inference", os.path.join(B, "news_inference.py"))
NI = importlib.util.module_from_spec(spec_); spec_.loader.exec_module(NI)
NI.load(B)
demo = NI.analyze_item(text="RT @KHOU: Roof torn off homes in Rockport, Texas after #HurricaneHarvey - families need rescue https://t.co/x",
                       image=Image.open(DATA[next(t for t in DEPLOY if TASKS[t]['modality'] == 'image')]['test']['cache_path'].iloc[0])
                       if any(TASKS[t]['modality'] == 'image' for t in DEPLOY) else None)
print(json.dumps(demo, indent=1)[:2500])

### 10b. Location extraction — manual check sheet (gives the paper a real precision number)

There are no gold location labels in CrisisMMD. `case_study/ner_manual_check.csv` holds 60 random **test** tweets with the extracted locations. Fill the two Y/N columns (≈20 minutes for two people), save it back with the same name (or attach it as a Kaggle input), and re-run the next cell: it reports precision of extracted locations and the share of tweets where a location was missed. If the sheet is not filled, report location extraction as qualitative only.

In [ ]:
CHECK = os.path.join(D['case'], "ner_manual_check.csv")
src_task = 'text_informative' if 'text_informative' in DATA else next(t for t in DATA if TASKS[t]['modality'] == 'text')
if not os.path.exists(CHECK):
    smp = DATA[src_task]['test'].sample(min(60, len(DATA[src_task]['test'])), random_state=0)
    pd.DataFrame({'tweet_id': smp['tweet_id'], 'event_name': smp['event_name'], 'tweet_text': smp['tweet_text'],
                  'extracted_locations': [' | '.join(NI.extract_locations(t)) for t in smp['tweet_text']],
                  'all_extracted_correct (Y/N)': '', 'location_missed (Y/N)': ''}).to_csv(CHECK, index=False)
    print("wrote", CHECK, "- fill the Y/N columns and re-run this cell")
chk = pd.read_csv(CHECK).fillna('')
c1, c2 = 'all_extracted_correct (Y/N)', 'location_missed (Y/N)'
filled = chk[chk[c1].astype(str).str.upper().isin(['Y', 'N'])]
if len(filled):
    with_ext = filled[filled['extracted_locations'] != '']
    ner_eval = {'n_checked': len(filled),
                'tweet_level_precision': float((with_ext[c1].str.upper() == 'Y').mean()) if len(with_ext) else None,
                'share_tweets_with_missed_location': float((filled[c2].astype(str).str.upper() == 'Y').mean())}
    jdump(ner_eval, os.path.join(D['metrics'], "ner_manual_check_results.json")); print(ner_eval)
else:
    print("manual check not filled yet - location extraction stays qualitative")

---
## 11. Case study + demo replay feed (test-split items only)

For each case-study event (2017 events that also appear in xBD), all **test-split** tweets are passed through the bundle — the models never saw them. Items are ordered by their real posting time (decoded from the tweet id), which is exactly the "replay mode" feed for the demo: the arrival of items is simulated, every output is produced live by the trained models.

Outputs per event: `replay_feed_<event>.json` (+ the images it references), `case_study_<event>_locations.csv` (most-mentioned places, item counts, mean priority, share of severe-damage images, geocoded where possible) and a summary JSON for the paper's case-study section.

In [ ]:
def tweet_time(tid):
    try: return datetime.datetime.utcfromtimestamp(((int(tid) >> 22) + 1288834974657) / 1000).isoformat() + 'Z'
    except Exception: return None

img_tbl = pd.concat([RAW['informative'][s].assign(split=s) for s in SPLITS])
img_tbl = img_tbl[img_tbl['cache_path'].notna()]
test_tweets = img_tbl[img_tbl['split'] == 'test']
# never put a tweet into the feed that any model trained on
train_ids = set().union(*[set(DATA[t]['train']['tweet_id']) for t in DATA])
case_summary = {}
for ev in CASE_STUDY_EVENTS:
    g = test_tweets[(test_tweets['event_name'] == ev) & ~test_tweets['tweet_id'].isin(train_ids)]
    if g.empty:
        print(f"{ev}: not in data, skipped"); continue
    feed_dir = os.path.join(D['case'], f"feed_{ev}"); os.makedirs(os.path.join(feed_dir, 'images'), exist_ok=True)
    items = []
    for tid, tg in tqdm(list(g.groupby('tweet_id')), desc=f"case study {ev}"):
        first = tg.iloc[0]
        img_dst = os.path.join(feed_dir, 'images', f"{first['image_id']}.jpg")
        if not os.path.exists(img_dst): shutil.copy2(first['cache_path'], img_dst)
        r = NI.analyze_item(text=first['tweet_text'], image=img_dst, item_id=tid, timestamp=tweet_time(tid))
        r['image_file'] = os.path.relpath(img_dst, feed_dir); r['event_name'] = ev
        items.append(r)
    items.sort(key=lambda r: r['timestamp'] or '')
    jdump({'event': ev, 'mode': 'replay of archived CrisisMMD test-split items; outputs from the trained models',
           'n_items': len(items), 'items': items}, os.path.join(feed_dir, f"replay_feed_{ev}.json"))

    loc_rows = {}
    for r in items:
        sev = r.get('image_damage', {}).get('label') == 'severe_damage'
        for l in r.get('locations', []):
            d = loc_rows.setdefault(l['name'], {'location': l['name'], 'n_items': 0, 'priority_sum': 0.0, 'n_severe_img': 0})
            d['n_items'] += 1; d['priority_sum'] += r['priority']['score']; d['n_severe_img'] += int(sev)
    locs = pd.DataFrame(loc_rows.values())
    if len(locs):
        locs['mean_priority'] = (locs['priority_sum'] / locs['n_items']).round(4)
        locs['share_severe_images'] = (locs['n_severe_img'] / locs['n_items']).round(4)
        locs = locs.drop(columns='priority_sum').sort_values(['n_items', 'mean_priority'], ascending=False)
        if GEOCODE:
            geo = [NI.geocode(n) for n in locs['location'].head(15)]
            locs.loc[locs.index[:15], 'lat'] = [x['lat'] if x else None for x in geo]
            locs.loc[locs.index[:15], 'lon'] = [x['lon'] if x else None for x in geo]
    locs.to_csv(os.path.join(D['case'], f"case_study_{ev}_locations.csv"), index=False)
    def _share(key, lab): return float(np.mean([r.get(key, {}).get('label') == lab for r in items])) if items else None
    case_summary[ev] = {
        'n_test_items': len(items),
        'time_span': [items[0]['timestamp'], items[-1]['timestamp']] if items else None,
        'share_text_informative': _share('text_informative', 'informative'),
        'image_damage_distribution': pd.Series([r.get('image_damage', {}).get('label') for r in items]).value_counts(normalize=True).round(4).to_dict(),
        'text_humanitarian_distribution': pd.Series([r.get('text_humanitarian', {}).get('label') for r in items]).value_counts(normalize=True).round(4).to_dict(),
        'share_items_with_location': float(np.mean([len(r.get('locations', [])) > 0 for r in items])),
        'top_locations': locs.head(10).to_dict('records') if len(locs) else [],
    }
    print(f"{ev}: {len(items)} items | with location {case_summary[ev]['share_items_with_location']:.1%} | "
          f"top places {list(locs['location'].head(5)) if len(locs) else []}")
jdump(case_summary, os.path.join(D['case'], "case_study_summary.json"))

---
## 12. Consolidated summary, manifest and ZIP

In [ ]:
print("=" * 78); print("  NEWS / SOCIAL-MEDIA MODULE - TEST RESULTS"); print("=" * 78)
print(summ.to_string(index=False))
print("\ntext label source:", json.dumps(TEXT_LABEL_SOURCE, indent=1))
print("\nleakage control:", json.dumps({t: {k: v for k, v in r.items() if 'dropped' in k} for t, r in LEAK.items()}, indent=1))

env_info = {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
            'transformers': transformers.__version__, 'timm': timm.__version__, 'sklearn': sklearn.__version__,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'data_source': DATA_SOURCE,
            'hyperparameters': HYPERPARAMS}
jdump(env_info, os.path.join(D['repro'], "environment.json"))

files = []
for root, _, fs in os.walk(OUT):
    for fn in sorted(fs):
        if fn.endswith('.tmp'): continue
        full = os.path.join(root, fn)
        files.append({'path': os.path.relpath(full, OUT), 'bytes': os.path.getsize(full)})
jdump({'n_files': len(files), 'total_bytes': sum(f['bytes'] for f in files), 'files': files}, os.path.join(OUT, "run_manifest.json"))

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_PATH = f"{OUT.rstrip('/')}_{stamp}.zip"
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as zf:
    for root, _, fs in os.walk(OUT):
        for fn in sorted(fs):
            full = os.path.join(root, fn)
            comp = zipfile.ZIP_STORED if fn.endswith(('.pth', '.bin', '.safetensors', '.jpg', '.png', '.npz')) else zipfile.ZIP_DEFLATED
            zf.write(full, os.path.join("crisis_news_results", os.path.relpath(full, OUT)), compress_type=comp)
with zipfile.ZipFile(ZIP_PATH) as zf:
    assert zf.testzip() is None
print(f"ZIP OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB, {len(files)} files)")